# Notebook 05 — Feature Engineering

**Main task:** dự đoán `target_popularity` bằng supervised regression.  
**Input:** `5.DATA/processed/ml_ready_dataset.parquet` (586.672 track thật).  
**Output:** engineered dataset, validation table, feature contract và train-only statistics cho Notebook 06.

Notebook này tạo **13 engineered features bằng code thật**. Clustering không được dùng như một feature của regression. Các thống kê theo thời kỳ và ngưỡng duration chỉ được fit trên train (`release_year < 2019`), sau đó transform train/test/inference bằng cùng statistics.

## 1. Mục tiêu và câu hỏi

1. Các giả thuyết từ EDA được chuyển thành column thật như thế nào?
2. 13 feature có tồn tại, đúng dtype, không missing và không infinite không?
3. Feature category sẽ được encoding thật ở bước training như thế nào?
4. Làm sao tránh leakage với `energy_vs_period_avg`, `dance_vs_period_avg` và duration thresholds?

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "src").exists() and (candidate / "5.DATA").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))

from src.features import (
    BASELINE_MODEL_FEATURES,
    ENGINEERED_CATEGORICAL_FEATURES,
    EXPECTED_ENGINEERED_FEATURES,
    MODEL_FEATURES,
    RAW_INPUT_FEATURES,
    TARGET,
    TEST_START_YEAR,
    FeatureBuilder,
    build_feature_contract,
    validate_engineered_features,
)

DATA_PATH = ROOT / "5.DATA" / "processed" / "ml_ready_dataset.parquet"
OUTPUT_DIR = ROOT / "5.DATA" / "processed"
FE_OUTPUT_DIR = ROOT / "7.ML" / "7.6.feature_engineering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Real dataset: {DATA_PATH}")

Project root: <PROJECT_ROOT>/hitradar-main
Real dataset: <PROJECT_ROOT>/5.DATA/processed/ml_ready_dataset.parquet


## 2. Đọc dữ liệu thật và time-based split

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không có real dataset: {DATA_PATH}. Không dùng synthetic fallback.")

df_raw = pd.read_parquet(DATA_PATH)
required = [*RAW_INPUT_FEATURES, TARGET, "track_id"]
missing_raw = [column for column in required if column not in df_raw.columns]
assert not missing_raw, f"Missing source columns: {missing_raw}"

shape_before = df_raw.shape
train_mask = df_raw["release_year"] < TEST_START_YEAR
assert train_mask.any() and (~train_mask).any(), "Time split tạo train/test rỗng."

print(f"Shape trước Feature Engineering: {shape_before}")
print(f"Train (< {TEST_START_YEAR}): {train_mask.sum():,}")
print(f"Test  (>= {TEST_START_YEAR}): {(~train_mask).sum():,}")
display(df_raw[["track_id", TARGET, *RAW_INPUT_FEATURES[:6]]].head())

Shape trước Feature Engineering: (586672, 20)
Train (< 2019): 554,547
Test  (>= 2019): 32,125


,track_id,target_popularity,duration_min,explicit,release_year,release_month,release_precision,danceability
0,0HOqINudNgQFpg1le5Hnqe,62,5.1767,True,1992,11.0,day,0.785
1,2Mb3zpobD0CvJGWv6NpsPy,62,4.7782,False,1992,5.0,day,0.761
2,2WElktskrNJEwgpp5Vouxk,62,2.8411,False,1992,1.0,day,0.683
3,6FHXJu99aQAUnivEjPhuUR,62,3.5333,False,1992,1.0,day,0.526
4,1QbQL5m30YNvukitIqAnFG,62,5.3421,False,1992,1.0,day,0.686


## 3. Fit trên TRAIN → transform toàn bộ dữ liệu

`FeatureBuilder.fit()` học duration quantiles và decade-level means từ train. Target không được truyền vào feature builder. Những decade không xuất hiện trong train sẽ dùng global train mean, không dùng test/future statistics.

In [3]:
feature_builder = FeatureBuilder(include_engineered=True)
feature_builder.fit(df_raw.loc[train_mask, RAW_INPUT_FEATURES])

engineered_matrix = feature_builder.transform(df_raw[RAW_INPUT_FEATURES])
df = pd.concat(
    [df_raw[["track_id", TARGET]].reset_index(drop=True), engineered_matrix.reset_index(drop=True)],
    axis=1,
)

shape_after = df.shape
print(f"Shape trước: {shape_before}")
print(f"Shape sau:   {shape_after}")
print(f"Số engineered features: {len(EXPECTED_ENGINEERED_FEATURES)}")
display(df[["track_id", TARGET, *EXPECTED_ENGINEERED_FEATURES]].head(8))

Shape trước: (586672, 20)
Shape sau:   (586672, 33)
Số engineered features: 13


,track_id,target_popularity,key_sin,key_cos,dance_energy,positive_energy,acoustic_energy_balance,dance_valence,acoustic_instrumental,tempo_energy,energy_vs_period_avg,dance_vs_period_avg,mood_quadrant,duration_category,tempo_category
0,0HOqINudNgQFpg1le5Hnqe,62,0.500000,8.660254e-01,0.622505,0.519415,0.217176,0.514175,1.579600e-01,70.499286,0.218618,0.212911,high_energy_positive,long,slow
1,2Mb3zpobD0CvJGWv6NpsPy,62,0.866025,5.000000e-01,0.519763,0.458293,0.171116,0.510631,0.000000e+00,69.520521,0.108618,0.188911,high_energy_positive,long,moderate
2,2WElktskrNJEwgpp5Vouxk,62,-0.500000,8.660254e-01,0.504737,0.617804,0.000066,0.570988,5.990200e-07,90.241507,0.164618,0.110911,high_energy_positive,short,fast
3,6FHXJu99aQAUnivEjPhuUR,62,1.000000,6.123234e-17,0.232492,0.125528,0.376586,0.149384,1.465830e-05,57.926310,-0.132382,-0.046089,calm_dark,standard,fast
4,1QbQL5m30YNvukitIqAnFG,62,-0.866025,5.000000e-01,0.417774,0.437871,0.297578,0.493234,3.870000e-05,77.791224,0.034618,0.113911,high_energy_positive,long,fast
5,1o53HbxmOy5TzThJdBaDZb,62,0.000000,1.000000e+00,0.335916,0.233748,0.004706,0.294903,1.446920e-07,59.419464,-0.058382,0.078911,high_energy_dark,long,moderate
6,7w4ojcH8NJ4LBmJZhSBTcT,62,-1.000000,-1.836970e-16,0.612850,0.742050,0.182692,0.629433,7.334000e-04,120.769700,0.275618,0.148911,high_energy_positive,long,fast
7,3WSyYBhLZRLbQo2tJgFvSR,62,0.500000,-8.660254e-01,0.494400,0.617176,0.171859,0.449400,3.847500e-05,79.165800,0.249618,0.027911,high_energy_positive,long,moderate


## 4. Validation bắt buộc

In [4]:
missing_features = [
    feature for feature in EXPECTED_ENGINEERED_FEATURES
    if feature not in df.columns
]

assert len(missing_features) == 0, f"Missing engineered features: {missing_features}"
assert len(EXPECTED_ENGINEERED_FEATURES) >= 12

feature_validation = validate_engineered_features(df)
assert feature_validation["Status"].eq("PASS").all(), feature_validation
display(feature_validation)

print("Engineered columns:")
print(EXPECTED_ENGINEERED_FEATURES)
print()
print("Missing values:")
display(df[EXPECTED_ENGINEERED_FEATURES].isna().sum().to_frame("missing_count"))

,Feature,Exists,Dtype,Missing Count,Infinite Count,Status
0,key_sin,True,float64,0,0,PASS
1,key_cos,True,float64,0,0,PASS
2,dance_energy,True,float64,0,0,PASS
3,positive_energy,True,float64,0,0,PASS
4,acoustic_energy_balance,True,float64,0,0,PASS
5,dance_valence,True,float64,0,0,PASS
6,acoustic_instrumental,True,float64,0,0,PASS
7,tempo_energy,True,float64,0,0,PASS
8,energy_vs_period_avg,True,float64,0,0,PASS
9,dance_vs_period_avg,True,float64,0,0,PASS


Engineered columns:
['key_sin', 'key_cos', 'dance_energy', 'positive_energy', 'acoustic_energy_balance', 'dance_valence', 'acoustic_instrumental', 'tempo_energy', 'energy_vs_period_avg', 'dance_vs_period_avg', 'mood_quadrant', 'duration_category', 'tempo_category']

Missing values:


,missing_count
key_sin,0
key_cos,0
dance_energy,0
positive_energy,0
acoustic_energy_balance,0
dance_valence,0
acoustic_instrumental,0
tempo_energy,0
energy_vs_period_avg,0
dance_vs_period_avg,0


## 5. Feature contract và encoding contract

Ba cột string/category (`mood_quadrant`, `duration_category`, `tempo_category`) không được đưa raw vào estimator. Notebook 06 dùng `ColumnTransformer` + `OneHotEncoder(handle_unknown='ignore')` được fit trên train. `MODEL_FEATURES` dưới đây chỉ được duyệt sau khi các columns đã được tạo và validation PASS.

In [5]:
MODEL_FEATURES_ACTUAL = [feature for feature in MODEL_FEATURES if feature in df.columns]
assert MODEL_FEATURES_ACTUAL == MODEL_FEATURES
assert not set(ENGINEERED_CATEGORICAL_FEATURES).difference(df.columns)

contract = build_feature_contract()
learned_statistics = feature_builder.get_learned_statistics()

print(f"Baseline features: {len(BASELINE_MODEL_FEATURES)}")
print(f"Engineered features: {len(EXPECTED_ENGINEERED_FEATURES)}")
print(f"MODEL_FEATURES: {len(MODEL_FEATURES)}")
display(pd.DataFrame({"MODEL_FEATURES": MODEL_FEATURES}))
display(pd.DataFrame([learned_statistics]).drop(columns=["energy_period_means", "dance_period_means"]))

Baseline features: 18
Engineered features: 13
MODEL_FEATURES: 31


,MODEL_FEATURES
0,duration_min
1,release_year
2,danceability
3,energy
4,loudness
5,speechiness
6,acousticness
7,instrumentalness
8,liveness
9,valence


,duration_q33,duration_q67,global_energy_mean,global_dance_mean,fit_row_count
0,3.16,4.1112,0.536681,0.558171,554547


## 6. Save output thật cho Notebook 06

In [6]:
ENGINEERED_DATA_PATH = OUTPUT_DIR / "features_engineered.parquet"
VALIDATION_PATH = FE_OUTPUT_DIR / "hard_requirement_feature_validation.csv"
CONTRACT_PATH = FE_OUTPUT_DIR / "hard_requirement_feature_contract.json"
STATS_PATH = FE_OUTPUT_DIR / "hard_requirement_train_statistics.json"

df.to_parquet(ENGINEERED_DATA_PATH, index=False)
feature_validation.to_csv(VALIDATION_PATH, index=False)
CONTRACT_PATH.write_text(json.dumps(contract, indent=2, ensure_ascii=False), encoding="utf-8")
STATS_PATH.write_text(json.dumps(learned_statistics, indent=2, ensure_ascii=False), encoding="utf-8")

reloaded = pd.read_parquet(ENGINEERED_DATA_PATH)
assert len(reloaded) == len(df)
assert all(feature in reloaded.columns for feature in EXPECTED_ENGINEERED_FEATURES)
assert validate_engineered_features(reloaded)["Status"].eq("PASS").all()

print(f"Saved engineered dataset: {ENGINEERED_DATA_PATH}")
print(f"Saved validation:         {VALIDATION_PATH}")
print(f"Saved contract:           {CONTRACT_PATH}")
print(f"Saved train statistics:   {STATS_PATH}")
print(f"Reloaded shape: {reloaded.shape}")

Saved engineered dataset: <PROJECT_ROOT>/5.DATA/processed/features_engineered.parquet
Saved validation:         <PROJECT_ROOT>/7.ML/7.6.feature_engineering/hard_requirement_feature_validation.csv
Saved contract:           <PROJECT_ROOT>/7.ML/7.6.feature_engineering/hard_requirement_feature_contract.json
Saved train statistics:   <PROJECT_ROOT>/7.ML/7.6.feature_engineering/hard_requirement_train_statistics.json
Reloaded shape: (586672, 33)


## 7. Key findings, insight và handoff

**Finding:** 13/13 engineered columns được tạo thật và validation PASS; output giữ đúng 586.672 track.  
**Interpretation:** interaction, cyclic key, nonlinear category và period-relative signals đã trở thành dữ liệu executable thay vì danh sách ý tưởng.  
**Impact:** Notebook 06 có thể so sánh baseline với engineered set trên cùng time split, còn deployment có thể tái tạo features từ raw input.  
**Leakage control:** learned statistics chỉ fit trên các track trước 2019.  
**Handoff:** Notebook 06 phải đọc `features_engineered.parquet`, kiểm tra `MODEL_FEATURES`, fit encoder/scaler trên train, chạy Linear/RF/XGBoost và lưu metrics mới.